# segment-line-intersect-2d — worked example 1: Find where a diagonal segment meets a vertical line

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `segment-line-intersect-2d`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A segment from S0 to S1 and an infinite line through L0, L1 intersect when the parametric equation t*(S1-S0) - s*(L1-L0) = L0-S0 has a solution where t ∈ [0,1]. The parameter t gives the fraction along the segment; s gives the position along the line (unconstrained). Solving the 2×2 linear system with `torch.linalg.solve` gives both parameters at once.

## Worked solution

**Step 1 — Form the directions.** The segment direction is `d = S1 - S0`. The line direction is `e = L1 - L0`. These are both 2-vectors.

**Step 2 — Build the 2×2 system.** The equation is `[d | -e] @ [t, s]^T = L0 - S0`. We stack `d` and `-e` as columns using `t.stack([d, -e], dim=1)`. The right-hand side is `b = L0 - S0`.

**Step 3 — Solve.** `ts = torch.linalg.solve(A, b)` gives `[t_seg, s_line]`.

**Step 4 — Hit test.** The intersection is on the segment only when `0 <= t_seg <= 1`. The line parameter s can be anything.

**Step 5 — Compute the intersection point.** The 2D intersection point is `S0 + t_seg * d`. We print it to visually verify.

In [ ]:
import torch as t

t.manual_seed(0)

def seg_line_intersect(S0, S1, L0, L1):
    d = S1 - S0
    e = L1 - L0
    A = t.stack([d, -e], dim=1)  # (2, 2): columns are d and -e
    b = L0 - S0
    ts = t.linalg.solve(A, b)
    t_seg = ts[0].item()
    s_line = ts[1].item()
    hit = (t_seg >= 0.0) and (t_seg <= 1.0)
    pt = S0 + t_seg * d
    return t_seg, s_line, hit, pt

# Segment from (0,0) to (4,2); line through (2,-1) and (2,3) (vertical x=2)
S0 = t.tensor([0.0, 0.0])
S1 = t.tensor([4.0, 2.0])
L0 = t.tensor([2.0, -1.0])
L1 = t.tensor([2.0,  3.0])

t_seg, s_line, hit, pt = seg_line_intersect(S0, S1, L0, L1)
print(f't_seg={t_seg:.4f}  s_line={s_line:.4f}  hit={hit}')
print(f'Intersection point: {pt.tolist()}')  # should be near (2, 1)